#  Feedback op jullie notebook
**Belangrijk:** lees deze feedback goed door. Per onderdeel staat wat goed gaat en wat jullie moeten verbeteren.
## Samenvatting
- Jullie recommender mist een **studentprofiel** → er worden geen echte aanbevelingen gedaan.
- De **EDA ontbreekt**, waardoor 35 punten uit de rubric niet worden gehaald....
- NLP wordt gestart maar niet afgemaakt (stopwoorden, cleaning, lemmatization).
- Cosine similarity wordt verkeerd toegepast (op dataset zelf, niet op profiel).
- Hybride model is incompleet en bevat fouten....

## Wat moet er zeker gebeuren
1. Voeg een studentprofiel toe en bereken similarity tegen alle modules.
2. Bouw EDA: dataset.info(), missing values, 3 grafieken.
3. Maak volledige NLP-cleaning pipeline.
4. Toon top-5 aanbevelingen voor minstens 2 testprofielen.
5. Leg per module uit waarom deze wordt aanbevolen (overlap keywords).

### **Imports:**
Hieronder zijn alle de packages die ik nodig heb om mijn recomonder model te bouwen.

In [59]:
import re
import nltk
from nltk.corpus import stopwords
import pandas as pd
from nltk.tokenize import word_tokenize, sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### **NLP:**

In [60]:
dataset = pd.read_csv('VKM_dataset_cleaned.csv')

#De kolom combined_text maken door alle tekstkolommen samen te voegen die nodig zijn voor elke rij. 
dataset['combined_text'] = (
    dataset['name'].fillna('') + ' ' +
    dataset['shortdescription'].fillna('') + ' ' +
    dataset['description'].fillna('') + ' ' +
    dataset['content'].fillna('') + ' ' +
    dataset['learningoutcomes'].fillna('') + ' ' +
    dataset['module_tags'].fillna('')
)

stop_words = stopwords.words("dutch")
lemma = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemma.lemmatize(t) for t in tokens]
    return " ".join(tokens)

dataset["combined_text"] = dataset["combined_text"].apply(clean_text)
dataset.head()


,id,name,shortdescription,description,content,studycredit,location,contact_id,level,learningoutcomes,module_tags,interests_match_score,popularity_score,estimated_difficulty,available_spots,start_date,status,combined_text
0,159,kennismaking met psychologie,"brein, gedragsbeinvloeding, ontwikkelingspsych...",in deze module leer je hoe je gedrag van jezel...,in deze module leer je hoe je gedrag van jezel...,15,Den Bosch,58,NLQF5,a. je beantwoordt vragen in een meerkeuze kenn...,"brein, gedragsbeinvloeding, ontwikkelingspsych...",0.54,319,1,79,2025-12-24,definitief,kennismaking psychologie brein gedragsbeinvloe...
1,160,learning and working abroad,"internationaal, persoonlijke ontwikkeling, ver...",studenten kiezen binnen de (stam) van de oplei...,studenten kiezen binnen de (stam) van de oplei...,15,Den Bosch,58,NLQF5,de student toont professioneel gedrag conform ...,"internationaal, persoonlijke, ontwikkeling, ve...",0.92,172,5,56,2025-12-20,definitief,learning and working abroad internationaal per...
2,161,proactieve zorgplanning,"proactieve zorgplanning, cocreatie, ziekenhuis",het jeroen bosch ziekenhuis wil graag samen me...,het jeroen bosch ziekenhuis wil graag samen me...,15,Den Bosch,59,NLQF5,de student past pro actieve zorgplanning toe b...,"proactieve, zorgplanning, cocreatie, ziekenhuis",0.78,217,5,55,2025-09-23,definitief,proactieve zorgplanning proactieve zorgplannin...
3,162,rouw en verlies,"rouw & verlies, palliatieve zorg & redeneren, ...",in deze module wordt stil gestaan bij rouw en ...,in deze module wordt stil gestaan bij rouw en ...,30,Den Bosch,58,NLQF6,de student regisseert en voert (deels) zelfsta...,"rouw, verlies, palliatieve, zorg, redeneren, t...",0.69,454,1,54,2025-10-25,definitief,rouw verlies rouw verlies palliatieve zorg red...
4,163,acuut complexe zorg,"acute zorg, complexiteit, ziekenhuis, revalidatie",in deze module kunnen studenten zich verdiepen...,in deze module kunnen studenten zich verdiepen...,30,Den Bosch,58,NLQF6,de student regisseert en voert (deels) zelfsta...,"acute, zorg, complexiteit, ziekenhuis, revalid...",0.40,178,5,38,2025-11-19,definitief,acuut complexe zorg acute zorg complexiteit zi...


### **Hybride Model:**

In [61]:

tfidf = TfidfVectorizer(max_features=6000, ngram_range=(1, 2),)
X = tfidf.fit_transform(dataset["combined_text"])
X.shape

(211, 6000)

In [62]:
from dataclasses import dataclass

@dataclass
#we kunnen dit uitbreiden met bv locatie en daarvan een conditie van maken.
class StudentProfile:
        #interesse van de student
    interests: str = ""
     #Wat is de doel die je wilt halen bij deze vkm
    goals: str = ""


In [63]:
def profile_vector(profile):
    text = profile.interests + " " + profile.goals
    text = clean_text(text)
    return tfidf.transform([text])

In [64]:
def recommend(profile, k=5):
    p_vec = profile_vector(profile)
    sims = cosine_similarity(p_vec, X).flatten()

    dataset_copy = dataset.copy()
    dataset_copy["content_sim"] = sims

    # Populariteit normaliseren
    pop = dataset_copy["popularity_score"].astype(float)
    pop = (pop - pop.min()) / (pop.max() - pop.min() + 1e-9)
    dataset_copy["popularity_norm"] = pop

    # Hybride recommendation model
    dataset_copy["Hybride_score"] = (
        0.75 * dataset_copy["content_sim"] +
        0.25 * dataset_copy["popularity_norm"]
    )

    return dataset_copy.sort_values("Hybride_score", ascending=False).head(k)

### **Test:**

In [65]:
profile = StudentProfile(
    interests="ik ben geintereseerd in koken",
    goals="leren koken, zodat ik in de toekomst voor mezelf en andere lekker kan koken"
)

recs = recommend(profile, k=5)
recs

,id,name,shortdescription,description,content,studycredit,location,contact_id,level,learningoutcomes,...,interests_match_score,popularity_score,estimated_difficulty,available_spots,start_date,status,combined_text,content_sim,popularity_norm,Hybride_score
72,254,module 2.1/2.3 toekomstgericht ontwerpen,"natuurinclusief ontwerpen, post humaan denken,...",studenten maken kennis met natuurinclusief ont...,studenten maken kennis met natuurinclusief ont...,15,Den Bosch en Tilburg,97,NLQF5,studenten leren om de ontwerpopgaven voor de t...,...,0.30,490,5,69,2025-11-26,definitief,module 2 1 2 3 toekomstgericht ontwerpen natuu...,0.057829,0.979592,0.288270
195,382,veranderen is mensenwerk,oud model van economische groei,als professional die veranderingsprocessen kan...,als professional die veranderingsprocessen kan...,30,Den Bosch,119,NLQF6,NaN,...,0.53,483,5,36,2025-09-05,voorlopig,veranderen mensenwerk oud model economische gr...,0.048129,0.965306,0.277423
158,345,chemie & gezondheid,"medicinale chemie, organische chemie, bioanaly...",chemie & gezondheid (werktitel) gaat over voed...,chemie & gezondheid (werktitel) gaat over voed...,15,Breda en Den Bosch,101,NLQF5,NaN,...,0.45,500,1,34,2025-12-04,voorlopig,chemie gezondheid medicinale chemie organische...,0.000000,1.000000,0.250000
101,283,safety first,"sociale veiligheid, fysieke veiligheid, risico's",het organiseren van sociale en fysieke veilige...,het organiseren van sociale en fysieke veilige...,15,Den Bosch,101,NLQF5,het organiseren van sociale en fysieke veilige...,...,0.71,499,5,74,2025-10-09,definitief,safety first sociale veiligheid fysieke veilig...,0.000000,0.997959,0.249490
85,267,module 4.1/4.2: drowning cities (bestaande minor),"ruimtelijk ontwerp, globale en nationale water...",an interdisciplinary team of student (urban pl...,an interdisciplinary team of student (urban pl...,30,Tilburg,94,NLQF6,1. developing skills in design and research an...,...,0.51,499,5,30,2025-09-23,definitief,module 4 1 4 2 drowning city bestaande minor r...,0.000000,0.997959,0.249490


In [66]:
profile2 = StudentProfile(
    interests="ik ben geintereseerd in informatica",
    goals="beter worden in coderen en een carieren mee opbouwen"
)

recs2 = recommend(profile2, k=5)
recs2

,id,name,shortdescription,description,content,studycredit,location,contact_id,level,learningoutcomes,...,interests_match_score,popularity_score,estimated_difficulty,available_spots,start_date,status,combined_text,content_sim,popularity_norm,Hybride_score
62,234,avans innovative studio basis,"persoonlijke ontwikkeling, interdisciplinair, ...",tijdens deze module leer je welke kennis en va...,tijdens deze module leer je welke kennis en va...,15,Breda,92,NLQF5,de student demonstreert persoonlijke groei op ...,...,0.84,389,5,25,2025-12-22,definitief,avans innovative studio basis persoonlijke ont...,0.082120,0.773469,0.254957
192,379,creative ai,"ai, creatieve sector, makerschap, originalitei...","gezamenlijk onderzoek naar creatieve, artistie...","gezamenlijk onderzoek naar creatieve, artistie...",30,Breda en Den Bosch,116,NLQF6,analyseren en interpreteren van de rol en impa...,...,0.89,448,1,23,2025-10-21,definitief,creative ai ai creatieve sector makerschap ori...,0.041518,0.893878,0.254608
158,345,chemie & gezondheid,"medicinale chemie, organische chemie, bioanaly...",chemie & gezondheid (werktitel) gaat over voed...,chemie & gezondheid (werktitel) gaat over voed...,15,Breda en Den Bosch,101,NLQF5,NaN,...,0.45,500,1,34,2025-12-04,voorlopig,chemie gezondheid medicinale chemie organische...,0.000000,1.000000,0.250000
101,283,safety first,"sociale veiligheid, fysieke veiligheid, risico's",het organiseren van sociale en fysieke veilige...,het organiseren van sociale en fysieke veilige...,15,Den Bosch,101,NLQF5,het organiseren van sociale en fysieke veilige...,...,0.71,499,5,74,2025-10-09,definitief,safety first sociale veiligheid fysieke veilig...,0.000000,0.997959,0.249490
85,267,module 4.1/4.2: drowning cities (bestaande minor),"ruimtelijk ontwerp, globale en nationale water...",an interdisciplinary team of student (urban pl...,an interdisciplinary team of student (urban pl...,30,Tilburg,94,NLQF6,1. developing skills in design and research an...,...,0.51,499,5,30,2025-09-23,definitief,module 4 1 4 2 drowning city bestaande minor r...,0.000000,0.997959,0.249490
